# SICP Exercise 1.6 — 用 Python 验证 `new-if` 的 bug

本 notebook 用 Python 复现 SICP Exercise 1.6 里的现象。Python 和 Scheme 一样是**应用序**求值,所以能完美演示这个 bug。

**核心对比**:
- `new_if` —— 普通函数 (模拟 Scheme 里用 cond 自定义的 new-if)
- Python 原生 `if` —— 特殊语法 (等价于 Scheme 的 if 特殊形式)

一个会死,一个能跑。

## 1. 准备工作 — 辅助函数

复刻 SICP 里的几个辅助过程。

In [1]:
def square(x):
    return x * x

def average(x, y):
    return (x + y) / 2

def improve(guess, x):
    return average(guess, x / guess)

def good_enough(guess, x):
    return abs(square(guess) - x) < 0.001

print('辅助函数已定义')

辅助函数已定义


## 2. Eva 的 `new_if` — 一个普通函数

在 Scheme 里她这样写:
```scheme
(define (new-if predicate then-clause else-clause)
  (cond (predicate then-clause)
        (else else-clause)))
```

Python 等价:

In [2]:
def new_if(predicate, then_clause, else_clause):
    if predicate:
        return then_clause
    else:
        return else_clause

## 3. Eva 的简单测试 — 看起来完全正常

Eva 用简单值测试 `new_if`,行为和 `if` 一模一样,**因此她以为成功了**。

In [3]:
print('new_if(2 == 3, 0, 5) =', new_if(2 == 3, 0, 5))    # 期望 5
print('new_if(1 == 1, 0, 5) =', new_if(1 == 1, 0, 5))    # 期望 0

new_if(2 == 3, 0, 5) = 5
new_if(1 == 1, 0, 5) = 0


## 4. Alyssa 把 `new_if` 用进 `sqrt_iter` — 这里开始出事

Alyssa 改写成:
```scheme
(define (sqrt-iter guess x)
  (new-if (good-enough? guess x)
          guess
          (sqrt-iter (improve guess x) x)))
```

注意: 第三个参数是**递归调用 sqrt_iter 自己**。

先设一个低递归深度限制,这样错误会很快显现(不然要等很久才耗尽默认的 1000 层)。

In [4]:
import sys
sys.setrecursionlimit(200)  # 降低递归深度,快速看到错误

def sqrt_iter_buggy(guess, x):
    return new_if(
        good_enough(guess, x),
        guess,
        sqrt_iter_buggy(improve(guess, x), x)   # ← 参数3:递归调用
    )

try:
    result = sqrt_iter_buggy(1.0, 9)
    print(f'√9 = {result}')
except RecursionError as e:
    print('❌ RecursionError: 栈溢出')
    print()
    print('原因:')
    print('  new_if 是普通函数 → 应用序要求先把三个参数全部算成值')
    print('  第三个参数是 sqrt_iter_buggy(...) 的递归调用')
    print('  每一层都卡在"算参数3",永远到不了 new_if 内部的 if 判断')
    print('  栈一直长一直长,直到触顶')

❌ RecursionError: 栈溢出

原因:
  new_if 是普通函数 → 应用序要求先把三个参数全部算成值
  第三个参数是 sqrt_iter_buggy(...) 的递归调用
  每一层都卡在"算参数3",永远到不了 new_if 内部的 if 判断
  栈一直长一直长,直到触顶


## 5. 正确版本 — 用 Python 原生 `if` (类比 Scheme 的特殊形式)

`if` 是 Python 语法,不是函数调用。遇到 `if 条件:` 时,Python 只会执行选中的那一支,**未选中那支根本不会被求值**。

这正是 Scheme 里 `if` 作为特殊形式的行为。

In [5]:
sys.setrecursionlimit(1000)  # 恢复默认

def sqrt_iter_correct(guess, x):
    if good_enough(guess, x):
        return guess
    else:
        return sqrt_iter_correct(improve(guess, x), x)

print('√9   =', sqrt_iter_correct(1.0, 9))
print('√2   =', sqrt_iter_correct(1.0, 2))
print('√100 =', sqrt_iter_correct(1.0, 100))
print('√1e6 =', sqrt_iter_correct(1.0, 1_000_000))

√9   = 3.00009155413138
√2   = 1.4142156862745097
√100 = 10.000000000139897
√1e6 = 1000.0000000000118


## 6. 看看实际收敛多快

牛顿法二次收敛,几层递归就能得到高精度结果。

In [6]:
def sqrt_iter_counted(guess, x, depth=0):
    if good_enough(guess, x):
        print(f'  深度 {depth} 层收敛,结果 {guess}')
        return guess
    return sqrt_iter_counted(improve(guess, x), x, depth + 1)

for n in [9, 2, 100, 10000, 1_000_000]:
    print(f'计算 √{n}:')
    sqrt_iter_counted(1.0, n)
    print()

计算 √9:
  深度 4 层收敛,结果 3.00009155413138

计算 √2:
  深度 3 层收敛,结果 1.4142156862745097

计算 √100:
  深度 7 层收敛,结果 10.000000000139897

计算 √10000:
  深度 10 层收敛,结果 100.00000025490743

计算 √1000000:
  深度 14 层收敛,结果 1000.0000000000118



## 7. 深层陷阱 — 即使不崩,`new_if` 也跟 `if` 不等价

就算参数不是递归(不会死循环),`new_if` 仍然**两支都会被求值**。用带副作用的函数做标记,能直观看到。

In [8]:
def expensive(label):
    print(f'  [已执行: {label}]')
    return label

print('--- 用原生 if (谓词为真) ---')
result = expensive('被选中的A') if True else expensive('不该执行的B')
print(f'返回: {result}')
print()

print('--- 用 new_if (谓词为真) ---')
result = new_if(True, expensive('被选中的A'), expensive('不该执行的B'))
print(f'返回: {result}')
print('    ↑ new_if 把"不该执行的B"也算了! 虽然最后没用它')

--- 用原生 if (谓词为真) ---
  [已执行: 被选中的A]
返回: 被选中的A

--- 用 new_if (谓词为真) ---
  [已执行: 被选中的A]
  [已执行: 不该执行的B]
返回: 被选中的A
    ↑ new_if 把"不该执行的B"也算了! 虽然最后没用它


## 8. 核心结论

| 维度 | 原生 `if` | `new_if` |
|---|---|---|
| 本质 | 语法 (特殊形式) | 普通函数 |
| 参数求值 | 只算谓词+选中那支 | 三个参数**全部**先算 |
| 递归终止 | 可以(跳过递归分支) | 不可以(递归分支被强行展开) |
| 副作用 | 只在选中那支触发 | 两支都会触发 |

**关键洞察**: 普通函数调用走应用序——参数必须先变成值才能进函数体。把 `cond` 包在普通函数里并不能拯救它,因为 `cond` 的特殊待遇要等函数体开始执行才生效,而程序已经死在"算参数"这一步了。

**结论**: `if` 必须作为特殊形式内置在语言里,不能用普通函数模拟——这是应用序语言的语法必然性。